In [ ]:
!pip install langchain-groq langgraph langchain-community duckduckgo-search -q

In [ ]:
!pip install -U ddgs -q

In [ ]:
from google.colab import userdata
my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0,
    api_key=my_api_key
)

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

def multiply(a: int, b: int) -> int:
    """Multiplies two numbers."""
    print("⭐ multiply() called")
    return a * b

tools = [search, multiply]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]

def assistant_node(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: State) -> Literal["tools", END]:
    if state["messages"][-1].tool_calls:
        return "tools"
    return END

In [ ]:
# Without HITL — agent acts immediately, no pause
builder = StateGraph(State)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph_no_hitl = builder.compile(checkpointer=MemorySaver())

In [ ]:
# Agent runs the tool immediately — no chance to stop it
config = {"configurable": {"thread_id": "no_hitl"}}
result = graph_no_hitl.invoke(
    {"messages": [("user", "What is 952124 * 123991?")]},
    config=config
)
print(result["messages"][-1].content)
# Tool ran. You had no say.

In [ ]:
# Without HITL — agent acts immediately, no pause
builder = StateGraph(State)
builder.add_node("assistant", assistant_node)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

graph_hitl = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before = ['tools']
)

In [ ]:
# Agent runs the tool immediately — no chance to stop it
config = {"configurable": {"thread_id": "hitl_test"}}
result = graph_hitl.invoke(
    {"messages": [("user", "What is 952124 * 123991?")]},
    config=config
)
print(result["messages"][-1].tool_calls)

In [ ]:
# Human approves - call the invoke again with None
# None means : 'no new message, just continue from where you stopped'
final = graph_hitl.invoke(None, config=config)

final['messages'][-1].content

# If you don't call it again, then it is rejected

In [ ]:
# Approve or Reject with input() -> will type Yes or No

def run_with_approval(user_message: str, thread_id: str):
    config = {'configurable': {"thread_id": thread_id}}

    result = graph_hitl.invoke(
        {"messages": [("user", user_message)]},
        config = config
    )

    last = result['messages'][-1]

    if not last.tool_calls:
        print('Agent answered directly (no tool needed)')
        print(last.content)
        return

    else:
        tool_call = last.tool_calls[0]
        print('Agent wants to use a tool')
        print('Tool: ', tool_call['name'])
        print('Args: ', tool_call['args'])

        decision = input('Approve? (yes/no): ').strip().lower()

        if decision == 'yes':
            final = graph_hitl.invoke(None, config=config)
            print("\n✅ Approved. Agent continued.")
            print("Answer:", final["messages"][-1].content)
        else:
            print("\n❌ Rejected. Agent was stopped. No action taken.")

In [ ]:
run_with_approval('What is 12 * 99?', thread_id = 'approval_1')

In [ ]:
run_with_approval('What is 12 * 99?', thread_id = 'approval_reject')

# Reflection and Self-Critique Agent

In [ ]:
!pip install langchain-groq langgraph -q

In [ ]:
from google.colab import userdata
my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0,
    api_key=my_api_key
)

In [ ]:
# The core idea - Writer and Critic (Reviewer)

# Step 1: Writer generates something
writer_prompt = 'Write a 3 sentence explanation of what a neural network is.'

draft = llm.invoke(writer_prompt).content

print(draft)

In [ ]:
# Step 2: Critic looks at the draft and gives feedback
critic_prompt = f'''You are a harsh but fair critic. Read the draft below and give specific feedback on how to
                    improve it. Point out what is unclear, too vague or missing

                    Draft:
                    {draft}

                    Give feedback only. Do not rewrite it.'''

feedback = llm.invoke(critic_prompt).content

print(feedback)

In [ ]:
# Step 3: Writer improves the draft on the basis of the feedback given

improve_prompt = f'''You are a writer. You wrote a draft and received feedback.
                    Rewrite the draft by addressing all the feedback points.draft

                    Original draft:
                    {draft}

                    Feedback:
                    {feedback}

                    Write the improved version only.'''

improved = llm.invoke(improve_prompt).content

print(improved)

In [ ]:
# Build properly using Langgraph

from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END

In [ ]:
# State now tracks the draft and the feedback separately
class State(TypedDict):
    messages: Annotated[list, add_messages]
    draft: str
    feedback: str
    round_number: int

In [ ]:
def writer_node(state:State):
    round_num = state.get('round_number', 0)

    if round_num == 0:
        user_request = state['messages'][-1].content
        prompt = f'Write a clear 3-4 sentence explanation of: {user_request}'

    else:
        prompt = f'''Improve this draft based on the feedback below.
                     Write the improved version only. No preamble.

                     Draft:
                     {state['draft']}

                     Feedback:
                     {state['feedback']}'''


    draft = llm.invoke(prompt).content

    print(f"\n✏️  Writer (round {round_num + 1}):\n{draft}")

    return {'draft': draft, 'round_number': round_num + 1}

In [ ]:
def critic_node(state: State):
    draft = state['draft']

    prompt = f'''You are a strict critic. Read this draft carefully.

                Draft:
                {draft}

                If the draft is clear, accurate and complete - response with exactly: APPROVED

                If it needs improvement - respond with: NEEDS WORK: (then your specific feedback on one line)
                '''

    feedback = llm.invoke(prompt).content

    print(f"\n🔍 Critic (round {state['round_number']}):\n{feedback}")
    return {"feedback": feedback}

In [ ]:
def should_continue(state: State) -> Literal['writer', END]:
    feedback = state['feedback']
    round_number = state['round_number']

    if "APPROVED" in feedback or round_number >= 3:
        print("\n✅ Critic approved. Stopping.")
        return END

    else:
        print("\n🔁 Critic not satisfied. Sending back to writer.")
        return "writer"

In [ ]:
builder = StateGraph(State)

builder.add_node('writer', writer_node)
builder.add_node('critic', critic_node)

builder.add_edge(START, 'writer')
builder.add_edge('writer', 'critic')
builder.add_conditional_edges('critic', should_continue)

reflection_graph = builder.compile()

In [ ]:
reflection_graph

In [ ]:
# TEST ITTTTT

result = reflection_graph.invoke({
    "messages": [("user", "What is a large language model")],
    "draft": "",
    "feedback": "",
    "round_number": 0
})

print(result['draft'])

In [ ]:
# Challenge 1 (Easy)
# Change the writer's task from explanation writing to writing a tweet (max 280 characters) about any tech topic.
# The critic should check if it is under 280 characters and engaging. Run the reflection graph and see how it improves.

In [ ]:
def writer_node(state:State):
    round_num = state.get('round_number', 0)

    if round_num == 0:
        user_request = state['messages'][-1].content
        prompt = f'Write a tweet of maximum 280 characters: {user_request}'

    else:
        prompt = f'''Improve this draft based on the feedback below.
                     Write the improved version only. No preamble.

                     Draft:
                     {state['draft']}

                     Feedback:
                     {state['feedback']}'''


    draft = llm.invoke(prompt).content

    print(f"\n✏️  Writer (round {round_num + 1}):\n{draft}")

    return {'draft': draft, 'round_number': round_num + 1}

In [ ]:
def critic_node(state: State):
    draft = state['draft']

    prompt = f'''You are a strict critic. Read this draft carefully. Draft should be under 280 characters and engaging.

                Draft:
                {draft}

                If the draft is clear, accurate and complete - response with exactly: APPROVED

                If it needs improvement - respond with: NEEDS WORK: (then your specific feedback on one line)
                '''

    feedback = llm.invoke(prompt).content

    print(f"\n🔍 Critic (round {state['round_number']}):\n{feedback}")
    return {"feedback": feedback}

In [ ]:
def should_continue(state: State) -> Literal['writer', END]:
    feedback = state['feedback']
    round_number = state['round_number']

    if "APPROVED" in feedback or round_number >= 3:
        print("\n✅ Critic approved. Stopping.")
        return END

    else:
        print("\n🔁 Critic not satisfied. Sending back to writer.")
        return "writer"

In [ ]:
builder = StateGraph(State)

builder.add_node('writer', writer_node)
builder.add_node('critic', critic_node)

builder.add_edge(START, 'writer')
builder.add_edge('writer', 'critic')
builder.add_conditional_edges('critic', should_continue)

reflection_graph = builder.compile()

In [ ]:
# TEST ITTTTT

result = reflection_graph.invoke({
    "messages": [("user", "What is agentic AI")],
    "draft": "",
    "feedback": "",
    "round_number": 0
})

print(result['draft'])

In [ ]:
# Challenge 2 (Medium)
# Build a reflection graph for code writing.
# The writer writes a Python function.
# The critic checks if it has a docstring, handles edge cases, and is readable.
# Loop until the critic approves.

In [ ]:
def writer_node(state:State):
    round_num = state.get('round_number', 0)

    if round_num == 0:
        user_request = state['messages'][-1].content
        prompt = f'Write a python program on: {user_request}'

    else:
        prompt = f'''Improve this draft based on the feedback below.
                     Write the improved version only. No preamble.

                     Draft:
                     {state['draft']}

                     Feedback:
                     {state['feedback']}'''


    draft = llm.invoke(prompt).content

    print(f"\n✏️  Writer (round {round_num + 1}):\n{draft}")

    return {'draft': draft, 'round_number': round_num + 1}

In [ ]:
def critic_node(state: State):
    draft = state['draft']

    prompt = f'''You are a strict critic. Read this draft carefully. Draft should have docstring, handle edge cases and is readable.

                Draft:
                {draft}

                If the draft is clear, accurate and complete - response with exactly: APPROVED

                If it needs improvement - respond with: NEEDS WORK: (then your specific feedback on one line)
                '''

    feedback = llm.invoke(prompt).content

    print(f"\n🔍 Critic (round {state['round_number']}):\n{feedback}")
    return {"feedback": feedback}

In [ ]:
def should_continue(state: State) -> Literal['writer', END]:
    feedback = state['feedback']
    round_number = state['round_number']

    if "APPROVED" in feedback or round_number >= 3:
        print("\n✅ Critic approved. Stopping.")
        return END

    else:
        print("\n🔁 Critic not satisfied. Sending back to writer.")
        return "writer"

In [ ]:
builder = StateGraph(State)

builder.add_node('writer', writer_node)
builder.add_node('critic', critic_node)

builder.add_edge(START, 'writer')
builder.add_edge('writer', 'critic')
builder.add_conditional_edges('critic', should_continue)

reflection_graph = builder.compile()

In [ ]:
# TEST ITTTTT

result = reflection_graph.invoke({
    "messages": [("user", "Write a function that takes user name and password, then do validation checks")],
    "draft": "",
    "feedback": "",
    "round_number": 0
})

print(result['draft'])

In [ ]:
# Challenge 3 (Hard)
# Add a second critic — a "Simplicity Critic" that checks if the writing is easy to understand
# for a beginner. The draft must pass both critics (Content Critic AND Simplicity Critic)
# before it is accepted. If either rejects it, it goes back to the writer.

In [ ]:
# State now tracks the draft and the feedback separately
class State(TypedDict):
    messages: Annotated[list, add_messages]
    draft: str
    content_feedback: str
    simplicity_feedback: str
    round_number: int

In [ ]:
def writer_node(state:State):
    round_num = state.get('round_number', 0)

    if round_num == 0:
        user_request = state['messages'][-1].content
        prompt = f'Write a clear 3-4 sentence explanation of: {user_request}'

    else:
        prompt = f'''Improve this draft based on the feedback below.
                     Write the improved version only. No preamble.

                     Draft:
                     {state['draft']}

                     Content Feedback:
                     {state['content_feedback']}

                     Simplicity Feedback:
                     {state['simplicity_feedback']}'''


    draft = llm.invoke(prompt).content

    print(f"\n✏️  Writer (round {round_num + 1}):\n{draft}")

    return {'draft': draft, 'round_number': round_num + 1}

In [ ]:
def critic_node(state: State):
    draft = state['draft']

    content_prompt = f'''You are a strict critic. Read this draft carefully.

                Draft:
                {draft}

                If the draft is clear, accurate and complete - response with exactly: APPROVED

                If it needs improvement - respond with:Content NEEDS WORK: (then your specific feedback on one line)
                '''


    simplicity_prompt = f'''You are a strict critic. Read this draft carefully.

                Draft:
                {draft}

                If the draft is easily understandable to beginner - response with exactly: APPROVED

                If it can be written in more simple words - respond with:Simplicity NEEDS WORK: (then your specific feedback on one line)
                '''

    content_feedback = llm.invoke(content_prompt).content
    simplicity_feedback = llm.invoke(simplicity_prompt).content

    print(f"\n🔍 Critic (round {state['round_number']}):")
    print(content_feedback)
    print('-'*80)
    print(simplicity_feedback)
    return {"content_feedback": content_feedback, "simplicity_feedback": simplicity_feedback}

In [ ]:
def should_continue(state: State) -> Literal['writer', END]:
    content_feedback = state['content_feedback']
    simplicity_feedback = state['simplicity_feedback']
    round_number = state['round_number']

    if "APPROVED" in content_feedback and "APPROVED" in simplicity_feedback:
        print("\n✅ Critic approved. Stopping.")
        return END

    elif round_number >= 3:
        print("\n✅ Stopping. After round 3")
        return END

    else:
        print("\n🔁 Critic not satisfied. Sending back to writer.")
        return "writer"

In [ ]:
builder = StateGraph(State)

builder.add_node('writer', writer_node)
builder.add_node('critic', critic_node)

builder.add_edge(START, 'writer')
builder.add_edge('writer', 'critic')
builder.add_conditional_edges('critic', should_continue)

reflection_graph = builder.compile()

In [ ]:
# TEST ITTTTT

result = reflection_graph.invoke({
    "messages": [("user", "What is a large language model")],
    "draft": "",
    "content_feedback": "",
    "simplicity_feedback": "",
    "round_number": 0
})

print(result['draft'])